# SRGAN to TLB-VFI Comparison Wrapper

This Colab notebook benchmarks **super-resolution (SR) + video frame interpolation (TLB-VFI)** pipelines on Vimeo-90K triplets.

It was configured for paths in Drive, but must now be changed if running elsewhere
- `TLBVFI-main`:
- `sr_modular`:
- `sr_project_outputs`:
- Vimeo triplet sequences root:
- test-list index file:

What it does:

1. Reads Vimeo triplet indices from `tri_testlist.txt`, e.g. `00002/0923`.
2. Lets you choose the maximum number of triplets and the starting folder/index.
3. Finds all available **SRGAN** checkpoints.
4. Loads the pretrained **TLB-VFI** model.
5. Creates **x4 low-resolution (LR)** versions of `im1`, `im2`, and `im3`.
6. Super-resolves each LR frame with every available SRGAN generator.
7. Uses the super-resolved endpoint frames (`SR im1` and `SR im3`) to predict the middle frame with **TLB-VFI**.
8. Compares the outputs against the **HR ground-truth** triplet and saves:
   - per-triplet PSNR / SSIM metrics
   - summary CSV files
   - clearly labeled comparison strips
   - absolute-error heatmaps
9. Optionally runs a **native-upscale** section where frames are enlarged first, interpolated at the larger size, and then resized back to native resolution for evaluation.


## Notes before running

- The standard benchmark section is the main one to run across all 100 triplets.
- The native-upscale x4/x16 section is much heavier. It is supported, but x16 may hit Colab GPU memory limits depending on the SRGAN checkpoint and the current runtime.
- This notebook loads TLB-VFI **in-process** instead of shelling out to `interpolate_one.py`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path("path/to/root/in/drive")
TLBVFI_REPO = DRIVE_ROOT / "path" / "TLBVFI-main"
SR_MODULAR_REPO = DRIVE_ROOT / "path" / "sr_modular"
SR_OUTPUTS_ROOT = DRIVE_ROOT / "path" / "sr_project_outputs"

# IMPORTANT:
# TRIPLET_ROOT should be the Vimeo `sequences` directory, not one folder such as `sequences/00078`.
# Entries in tri_testlist.txt, e.g. `00002/0923`, are resolved relative to this directory.
TRIPLET_ROOT = DRIVE_ROOT / "DATA" / "vimeo_triplet" / "sequences"
TRIPLET_INDEX_FILE = DRIVE_ROOT / "DATA" / "vimeo_triplet" / "tri_testlist.txt"

RESULTS_ROOT = SR_OUTPUTS_ROOT / "vfi_comparisons"
STANDARD_RESULTS_ROOT = RESULTS_ROOT / "standard_x4"
NATIVE_RESULTS_ROOT = RESULTS_ROOT / "native_upscale"

CONFIG_PATH = TLBVFI_REPO / "configs" / "Template-LBBDM-video.yaml"

# Data selection controls.
# N_TRIPLETS is the maximum number of triplets to load from tri_testlist.txt after applying the start filter.
# Set START_FROM_FOLDER to a folder such as "00003" to start from the first test-list entry in that folder.
# Set START_FROM_INDEX to a full index such as "00003/0010" for a more precise start point.
# If START_FROM_INDEX is not None, it takes priority over START_FROM_FOLDER.
N_TRIPLETS = 5000
START_FROM_FOLDER = "00044"  # example: "00003"
START_FROM_INDEX = None    # example: "00003/0010"

STANDARD_DOWNSAMPLE_FACTOR = 4
SAVE_IMAGES = True
FORCE_RERUN = True

# Native upscale section: optional and heavier
RUN_NATIVE_UPSCALE_SECTION = False
NATIVE_TARGET_SCALES = [4, 16]
NATIVE_MAX_TRIPLETS = 10  # reduce if Colab VRAM is tight

for p in [RESULTS_ROOT, STANDARD_RESULTS_ROOT, NATIVE_RESULTS_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("Configured paths:")
print("TLBVFI_REPO        =", TLBVFI_REPO)
print("SR_MODULAR_REPO    =", SR_MODULAR_REPO)
print("SR_OUTPUTS_ROOT    =", SR_OUTPUTS_ROOT)
print("TRIPLET_ROOT       =", TRIPLET_ROOT)
print("TRIPLET_INDEX_FILE =", TRIPLET_INDEX_FILE)
print("RESULTS_ROOT       =", RESULTS_ROOT)
print("N_TRIPLETS         =", N_TRIPLETS)
print("START_FROM_FOLDER  =", START_FROM_FOLDER)
print("START_FROM_INDEX   =", START_FROM_INDEX)


In [ ]:
# Run this cell only if imports fail in the next cells.
# It is intentionally commented so it does not overwrite a working Colab runtime.
#
# %cd "{TLBVFI_REPO}"
# %pip install -q -r requirements.txt
# %pip install -q scikit-image pyyaml pandas tqdm matplotlib pillow

In [ ]:
# pip install pytorch_lightning
# pip install lpips
# pip install pytorch_fid

In [ ]:
import gc
import json
import math
import os
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchvision.transforms as T
import yaml

from PIL import Image
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# Make both repos importable. Put the TLB-VFI repo root FIRST so its local
# `datasets/` package wins over any installed package named `datasets`.
if str(TLBVFI_REPO) in sys.path:
    sys.path.remove(str(TLBVFI_REPO))
sys.path.insert(0, str(TLBVFI_REPO))

if str(SR_MODULAR_REPO.parent) not in sys.path:
    sys.path.insert(0, str(SR_MODULAR_REPO.parent))

# Ensure repo-local packages behave like regular packages in Colab
for init_path in [
    TLBVFI_REPO / "datasets" / "__init__.py",
    TLBVFI_REPO / "runners" / "__init__.py",
    TLBVFI_REPO / "model" / "__init__.py",
    TLBVFI_REPO / "configs" / "__init__.py",
]:
    init_path.touch(exist_ok=True)

# Clear potentially conflicting cached modules from earlier failed imports
for name in list(sys.modules.keys()):
    if (
        name == "datasets"
        or name == "utils"
        or name.startswith("datasets.")
        or name.startswith("runners.")
        or name.startswith("model.")
        or name.startswith("configs.")
    ):
        del sys.modules[name]

# Change into the repo so any relative file assumptions inside TLB-VFI behave normally
os.chdir(TLBVFI_REPO)

# sr_modular imports
from sr_modular.models import Generator
from sr_modular.utils import convert_image as sr_convert_image

# TLB-VFI imports
from utils import dict2namespace
from model.BrownianBridge.LatentBrownianBridgeModel import LatentBrownianBridgeModel

print("Imported sr_modular and TLB-VFI successfully.")
print("TLBVFI repo root:", TLBVFI_REPO)
print("datasets/custom.py exists:", (TLBVFI_REPO / "datasets" / "custom.py").exists())
print("sys.path[0]:", sys.path[0])


In [ ]:
# Optional diagnostic: run this only if the TLB-VFI import cell above still fails.
print("Repo contents check:")
for rel in ["utils.py", "datasets/custom.py", "runners/utils.py", "model/BrownianBridge/LatentBrownianBridgeModel.py"]:
    p = TLBVFI_REPO / rel
    print(f"{rel}: {'OK' if p.exists() else 'MISSING'} -> {p}")


In [ ]:
def find_first_file(name: str, search_roots):
    for root in search_roots:
        root = Path(root)
        if not root.exists():
            continue
        matches = list(root.rglob(name))
        if matches:
            return matches[0]
    return None


def read_triplet_index_file(index_file: Path) -> list[str]:
    """Read Vimeo triplet indices such as `00002/0923` from tri_testlist.txt."""
    index_file = Path(index_file)
    if not index_file.exists():
        raise FileNotFoundError(f"Triplet index file not found: {index_file}")

    entries: list[str] = []
    with index_file.open("r", encoding="utf-8") as f:
        for line_no, raw_line in enumerate(f, start=1):
            entry = raw_line.strip().replace("\\", "/")
            if not entry or entry.startswith("#"):
                continue

            parts = entry.split("/")
            if len(parts) != 2 or any(part == "" for part in parts):
                raise ValueError(
                    f"Invalid triplet index at {index_file}:{line_no}: {raw_line!r}. "
                    "Expected format like '00002/0923'."
                )
            entries.append(entry)

    if not entries:
        raise ValueError(f"No usable triplet entries found in {index_file}")

    return entries


def _slice_triplet_entries(
    entries: list[str],
    limit: int | None = 100,
    start_folder: str | None = None,
    start_index: str | None = None,
) -> list[str]:
    """Apply optional start and maximum-count filters to the test-list entries."""
    if start_index is not None:
        start_index = str(start_index).strip().replace("\\", "/")
        if start_index not in entries:
            raise ValueError(f"START_FROM_INDEX={start_index!r} was not found in the triplet index list.")
        start_pos = entries.index(start_index)
        entries = entries[start_pos:]

    elif start_folder is not None:
        start_folder = str(start_folder).strip().strip("/")
        folder_positions = [
            i for i, entry in enumerate(entries)
            if entry.split("/", 1)[0] == start_folder
        ]
        if not folder_positions:
            raise ValueError(f"START_FROM_FOLDER={start_folder!r} was not found in the triplet index list.")
        entries = entries[folder_positions[0]:]

    if limit is not None:
        limit = int(limit)
        if limit < 1:
            raise ValueError("limit must be >= 1, or None to load all available entries.")
        entries = entries[:limit]

    return entries


def triplet_index_from_dir(triplet_dir: Path, triplet_root: Path = TRIPLET_ROOT) -> str:
    """Return the Vimeo triplet index, e.g. `00002/0923`, for a triplet directory."""
    triplet_dir = Path(triplet_dir)
    triplet_root = Path(triplet_root)
    try:
        return triplet_dir.relative_to(triplet_root).as_posix()
    except ValueError:
        # Fallback for legacy paths that may already point directly to one triplet folder.
        return triplet_dir.name


def normalize_triplet_index(triplet_name: str | Path) -> str:
    """Accept `00002/0923` or output-safe `00002__0923` and normalize to `00002/0923`."""
    return str(triplet_name).strip().replace("\\", "/").replace("__", "/")


def triplet_output_name(triplet_name: str | Path) -> str:
    """Convert `00002/0923` to an output-safe folder name `00002__0923`."""
    return normalize_triplet_index(triplet_name).replace("/", "__")


def triplet_dir_from_index(triplet_name: str | Path, triplet_root: Path = TRIPLET_ROOT) -> Path:
    """Resolve a triplet index such as `00002/0923` relative to the sequences root."""
    return Path(triplet_root) / Path(normalize_triplet_index(triplet_name))


def list_triplet_dirs(
    triplet_root: Path,
    index_file: Path | None = None,
    limit: int | None = 100,
    start_folder: str | None = None,
    start_index: str | None = None,
) -> list[Path]:
    """
    Return triplet directories.

    With index_file provided, only indices listed in tri_testlist.txt are used.
    Entries such as `00002/0923` are resolved as:
        triplet_root / "00002" / "0923"

    start_folder examples:
        "00003"      -> start at the first `00003/...` entry in the file

    start_index examples:
        "00003/0010" -> start exactly at that full index
    """
    triplet_root = Path(triplet_root)
    if not triplet_root.exists():
        raise FileNotFoundError(f"Triplet root not found: {triplet_root}")

    # Legacy fallback: list direct child folders if no index file is supplied.
    if index_file is None:
        triplet_dirs = sorted([p for p in triplet_root.iterdir() if p.is_dir()])
        if len(triplet_dirs) == 0:
            raise ValueError(f"No triplet subdirectories found under {triplet_root}")
        if limit is not None:
            triplet_dirs = triplet_dirs[:int(limit)]
        return triplet_dirs

    entries = read_triplet_index_file(index_file)
    entries = _slice_triplet_entries(
        entries,
        limit=limit,
        start_folder=start_folder,
        start_index=start_index,
    )

    triplet_dirs = [triplet_root / Path(entry) for entry in entries]
    missing = [p for p in triplet_dirs if not p.is_dir()]
    if missing:
        examples = "\n".join(str(p) for p in missing[:5])
        raise FileNotFoundError(
            "Some triplet directories from the index file were not found. "
            "Make sure TRIPLET_ROOT points to the Vimeo `sequences` folder, not to a single sequence folder.\n"
            f"First missing paths:\n{examples}"
        )

    return triplet_dirs


def load_triplet_images(triplet_dir: Path):
    paths = [triplet_dir / "im1.png", triplet_dir / "im2.png", triplet_dir / "im3.png"]
    for p in paths:
        if not p.exists():
            raise FileNotFoundError(f"Missing frame file: {p}")
    return [Image.open(p).convert("RGB") for p in paths]


def pil_to_np_uint8(img: Image.Image) -> np.ndarray:
    return np.asarray(img, dtype=np.uint8)


def np_uint8_to_pil(arr: np.ndarray) -> Image.Image:
    arr = np.asarray(arr)
    if arr.dtype != np.uint8:
        arr = np.clip(np.round(arr), 0, 255).astype(np.uint8)
    return Image.fromarray(arr)


def save_pil(img: Image.Image, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    img.save(path)


def pil_downsample(img: Image.Image, factor: int) -> Image.Image:
    w, h = img.size
    if w % factor != 0 or h % factor != 0:
        raise ValueError(f"Image size {img.size} not divisible by factor={factor}")
    return img.resize((w // factor, h // factor), Image.BICUBIC)


def pil_resize(img: Image.Image, size_hw) -> Image.Image:
    h, w = size_hw
    return img.resize((w, h), Image.BICUBIC)


def pil_abs_diff_heatmap(pred: Image.Image, gt: Image.Image):
    pred_np = pil_to_np_uint8(pred).astype(np.float32)
    gt_np = pil_to_np_uint8(gt).astype(np.float32)
    if pred_np.shape != gt_np.shape:
        raise ValueError(f"Shape mismatch: pred={pred_np.shape}, gt={gt_np.shape}")
    diff = np.abs(pred_np - gt_np).mean(axis=2) / 255.0
    return diff


def compute_rgb_metrics(pred: Image.Image, gt: Image.Image):
    pred_np = pil_to_np_uint8(pred)
    gt_np = pil_to_np_uint8(gt)
    if pred_np.shape != gt_np.shape:
        raise ValueError(f"Shape mismatch: pred={pred_np.shape}, gt={gt_np.shape}")

    psnr = peak_signal_noise_ratio(gt_np, pred_np, data_range=255)
    ssim = structural_similarity(gt_np, pred_np, channel_axis=2, data_range=255)
    return float(psnr), float(ssim)


def recursive_passes_for_total_scale(total_scale: int, base_scale: int = 4) -> int:
    if total_scale < 1:
        raise ValueError("total_scale must be >= 1")
    if total_scale == 1:
        return 0
    p = 0
    current = 1
    while current < total_scale:
        current *= base_scale
        p += 1
    if current != total_scale:
        raise ValueError(f"total_scale={total_scale} is not an integer power of base_scale={base_scale}")
    return p


def clear_cuda():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()


In [ ]:
def discover_srgan_checkpoints(outputs_root: Path):
    outputs_root = Path(outputs_root)
    candidates = sorted(outputs_root.rglob("SRGAN_ResNet_*.pt"))

    best_by_name = {}
    for p in candidates:
        name = p.stem
        base_name = name.replace("_best", "")
        current = best_by_name.get(base_name)

        if current is None:
            best_by_name[base_name] = p
        else:
            if p.stem.endswith("_best") and not current.stem.endswith("_best"):
                best_by_name[base_name] = p

    if not best_by_name:
        raise FileNotFoundError(f"No SRGAN checkpoints found under: {outputs_root}")

    return dict(sorted(best_by_name.items()))


def load_srgan_generator(checkpoint_path: Path, device=device):
    checkpoint_path = Path(checkpoint_path)
    ckpt = torch.load(checkpoint_path, map_location="cpu")
    generator = Generator()
    generator.load_state_dict(ckpt["generator_state_dict"])
    generator = generator.to(device).eval()
    return generator


@torch.no_grad()
def srgan_super_resolve_pil(
    img: Image.Image,
    generator: Generator,
    recursive_passes: int = 1,
    device=device,
):
    if recursive_passes < 0:
        raise ValueError("recursive_passes must be >= 0")

    current = img
    for _ in range(recursive_passes):
        lr_tensor = sr_convert_image(current, source="pil", target="imagenet-norm").unsqueeze(0).to(device)
        sr_tensor = generator(lr_tensor).squeeze(0).detach().cpu()
        current = sr_convert_image(sr_tensor, source="[-1, 1]", target="pil")
        clear_cuda()
    return current


def bicubic_upscale_pil(img: Image.Image, total_scale: int):
    if total_scale < 1:
        raise ValueError("total_scale must be >= 1")
    if total_scale == 1:
        return img.copy()
    w, h = img.size
    return img.resize((w * total_scale, h * total_scale), Image.BICUBIC)

In [ ]:
def load_tlbvfi_model(
    repo_root: Path,
    config_path: Path,
    full_model_path: Path,
    vq_ckpt_path: Path | None = None,
    device=device,
):
    repo_root = Path(repo_root)
    config_path = Path(config_path)
    full_model_path = Path(full_model_path)

    if not config_path.exists():
        raise FileNotFoundError(f"Missing config file: {config_path}")
    if not full_model_path.exists():
        raise FileNotFoundError(f"Missing TLB-VFI model file: {full_model_path}")

    # TLB-VFI's own main.py uses yaml.load(..., Loader=yaml.FullLoader).
    # The config contains Python-specific tags like !!python/list and !!python/tuple,
    # so yaml.safe_load(...) will fail on this repo.
    with open(config_path, "r") as f:
        try:
            cfg = yaml.load(f, Loader=yaml.FullLoader)
        except Exception:
            f.seek(0)
            cfg = yaml.load(f, Loader=yaml.UnsafeLoader)

    cfg.setdefault("model", {})
    cfg["model"]["model_load_path"] = str(full_model_path)

    # Point the VQ checkpoint at the user's downloaded file when provided.
    model_cfg = cfg["model"]
    if (
        vq_ckpt_path is not None
        and Path(vq_ckpt_path).exists()
        and "VQGAN" in model_cfg
        and "params" in model_cfg["VQGAN"]
    ):
        model_cfg["VQGAN"]["params"]["ckpt_path"] = str(vq_ckpt_path)

    ns_cfg = dict2namespace(cfg)

    model = LatentBrownianBridgeModel(ns_cfg.model).to(device)

    # Manually load the pretrained TLB-VFI weights because we are bypassing the
    # repo's runner/checkpoint machinery and instantiating the model directly.
    ckpt = torch.load(full_model_path, map_location=device)

    if isinstance(ckpt, dict):
        if "state_dict" in ckpt:
            state_dict = ckpt["state_dict"]
        elif "model_state_dict" in ckpt:
            state_dict = ckpt["model_state_dict"]
        elif "model" in ckpt and isinstance(ckpt["model"], dict):
            state_dict = ckpt["model"]
        elif "net" in ckpt and isinstance(ckpt["net"], dict):
            state_dict = ckpt["net"]
        else:
            # Some checkpoints are already plain state_dict objects.
            state_dict = ckpt
    else:
        state_dict = ckpt

    cleaned_state_dict = {}
    for k, v in state_dict.items():
        new_key = k
        if new_key.startswith("module."):
            new_key = new_key[len("module."):]
        if new_key.startswith("net."):
            new_key = new_key[len("net."):]
        cleaned_state_dict[new_key] = v

    incompatible = model.load_state_dict(cleaned_state_dict, strict=False)
    model = model.eval()

    print("Loaded TLB-VFI checkpoint:", full_model_path)
    if hasattr(incompatible, "missing_keys") and incompatible.missing_keys:
        print("Missing keys (first 10):", incompatible.missing_keys[:10])
    if hasattr(incompatible, "unexpected_keys") and incompatible.unexpected_keys:
        print("Unexpected keys (first 10):", incompatible.unexpected_keys[:10])

    return model


_tlb_transform = T.Compose(
    [
        T.ToTensor(),
        T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ]
)


@torch.no_grad()
def tlbvfi_interpolate_pil(frame0: Image.Image, frame1: Image.Image, model, device=device) -> Image.Image:
    t0 = _tlb_transform(frame0).unsqueeze(0).to(device)
    t1 = _tlb_transform(frame1).unsqueeze(0).to(device)
    out = model.sample(t0, t1).squeeze(0).detach().cpu()
    out = (out / 2.0 + 0.5).clamp(0, 1)
    out_np = np.round(out.permute(1, 2, 0).numpy() * 255.0).astype(np.uint8)
    clear_cuda()
    return Image.fromarray(out_np)

In [ ]:
# Resolve TLB-VFI weight paths automatically.
VFI_FULL_MODEL = find_first_file("vimeo_unet.pth", [TLBVFI_REPO, DRIVE_ROOT])
VFI_VQ_CKPT = find_first_file("vimeo_new.ckpt", [TLBVFI_REPO, DRIVE_ROOT])

# Discover triplets from tri_testlist.txt and SRGAN checkpoints.
TRIPLET_DIRS = list_triplet_dirs(
    TRIPLET_ROOT,
    index_file=TRIPLET_INDEX_FILE,
    limit=N_TRIPLETS,
    start_folder=START_FROM_FOLDER,
    start_index=START_FROM_INDEX,
)
SRGAN_CHECKPOINTS = discover_srgan_checkpoints(SR_OUTPUTS_ROOT)

print("Resolved assets:")
print("  TLB full model:", VFI_FULL_MODEL)
print("  TLB VQ ckpt   :", VFI_VQ_CKPT)
print("  Triplets      :", len(TRIPLET_DIRS))
if TRIPLET_DIRS:
    print("  First triplet :", triplet_index_from_dir(TRIPLET_DIRS[0]))
    print("  Last triplet  :", triplet_index_from_dir(TRIPLET_DIRS[-1]))
print("  SRGAN models  :", len(SRGAN_CHECKPOINTS))
for name, path in SRGAN_CHECKPOINTS.items():
    print("   -", name, "->", path)


In [ ]:
TLB_MODEL = load_tlbvfi_model(
    repo_root=TLBVFI_REPO,
    config_path=CONFIG_PATH,
    full_model_path=VFI_FULL_MODEL,
    vq_ckpt_path=VFI_VQ_CKPT,
    device=device,
)
print("Loaded TLB-VFI model.")

In [ ]:
SRGAN_MODELS = {
    model_name: load_srgan_generator(path, device=device)
    for model_name, path in SRGAN_CHECKPOINTS.items()
}
print("Loaded SRGAN generators:", list(SRGAN_MODELS.keys()))

In [ ]:
def run_standard_x4_benchmark(
    triplet_dirs,
    srgan_models,
    tlb_model,
    results_root: Path,
    downsample_factor: int = 4,
    save_images: bool = True,
    force_rerun: bool = False,
):
    results_root = Path(results_root)
    csv_path = results_root / "per_triplet_metrics.csv"

    if csv_path.exists() and not force_rerun:
        print(f"Loading existing results from {csv_path}")
        return pd.read_csv(csv_path)

    rows = []

    for triplet_dir in tqdm(triplet_dirs, desc="Standard x4 benchmark"):
        triplet_name = triplet_index_from_dir(triplet_dir)
        output_name = triplet_output_name(triplet_name)

        hr1, hr2, hr3 = load_triplet_images(triplet_dir)
        gt_size_hw = (hr2.height, hr2.width)

        lr1 = pil_downsample(hr1, downsample_factor)
        lr2 = pil_downsample(hr2, downsample_factor)
        lr3 = pil_downsample(hr3, downsample_factor)

        # Bicubic baselines
        bic1 = pil_resize(lr1, gt_size_hw)
        bic2 = pil_resize(lr2, gt_size_hw)
        bic3 = pil_resize(lr3, gt_size_hw)
        bic_vfi_mid = tlbvfi_interpolate_pil(bic1, bic3, tlb_model)

        if save_images:
            save_dir = results_root / "BICUBIC" / output_name
            save_pil(bic1, save_dir / "im1_sr.png")
            save_pil(bic2, save_dir / "im2_sr.png")
            save_pil(bic3, save_dir / "im3_sr.png")
            save_pil(bic_vfi_mid, save_dir / "vfi_mid.png")

        psnr, ssim = compute_rgb_metrics(bic2, hr2)
        rows.append({
            "triplet": triplet_name,
            "variant": "BICUBIC",
            "pipeline": "direct_middle",
            "compare_mode": "same_resolution",
            "psnr": psnr,
            "ssim": ssim,
        })

        psnr, ssim = compute_rgb_metrics(bic_vfi_mid, hr2)
        rows.append({
            "triplet": triplet_name,
            "variant": "BICUBIC",
            "pipeline": "vfi_middle",
            "compare_mode": "same_resolution",
            "psnr": psnr,
            "ssim": ssim,
        })

        for model_name, generator in srgan_models.items():
            save_dir = results_root / model_name / output_name
            try:
                sr1 = srgan_super_resolve_pil(lr1, generator, recursive_passes=1, device=device)
                sr2 = srgan_super_resolve_pil(lr2, generator, recursive_passes=1, device=device)
                sr3 = srgan_super_resolve_pil(lr3, generator, recursive_passes=1, device=device)
                vfi_mid = tlbvfi_interpolate_pil(sr1, sr3, tlb_model, device=device)

                if save_images:
                    save_pil(sr1, save_dir / "im1_sr.png")
                    save_pil(sr2, save_dir / "im2_sr.png")
                    save_pil(sr3, save_dir / "im3_sr.png")
                    save_pil(vfi_mid, save_dir / "vfi_mid.png")

                psnr, ssim = compute_rgb_metrics(sr2, hr2)
                rows.append({
                    "triplet": triplet_name,
                    "variant": model_name,
                    "pipeline": "direct_middle",
                    "compare_mode": "same_resolution",
                    "psnr": psnr,
                    "ssim": ssim,
                })

                psnr, ssim = compute_rgb_metrics(vfi_mid, hr2)
                rows.append({
                    "triplet": triplet_name,
                    "variant": model_name,
                    "pipeline": "vfi_middle",
                    "compare_mode": "same_resolution",
                    "psnr": psnr,
                    "ssim": ssim,
                })

            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    clear_cuda()
                    rows.append({
                        "triplet": triplet_name,
                        "variant": model_name,
                        "pipeline": "oom",
                        "compare_mode": "same_resolution",
                        "psnr": np.nan,
                        "ssim": np.nan,
                    })
                    print(f"OOM at {triplet_name} / {model_name}.")
                    continue
                raise

    df = pd.DataFrame(rows)
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(csv_path, index=False)
    print(f"Saved per-triplet metrics to {csv_path}")
    return df


In [ ]:
standard_df = run_standard_x4_benchmark(
    triplet_dirs=TRIPLET_DIRS,
    srgan_models=SRGAN_MODELS,
    tlb_model=TLB_MODEL,
    results_root=STANDARD_RESULTS_ROOT,
    downsample_factor=STANDARD_DOWNSAMPLE_FACTOR,
    save_images=SAVE_IMAGES,
    force_rerun=FORCE_RERUN,
)

display(standard_df.loc[standard_df["pipeline"] == "vfi_middle"].head())


In [ ]:
PIPELINE_DISPLAY_NAMES = {
    "direct_middle": "SR middle frame (im2 only)",
    "vfi_middle": "SR + TLB-VFI predicted im2",
    "direct_middle_back_to_native": "Upscaled SR middle frame resized back to native",
    "vfi_middle_back_to_native": "Upscaled SR + TLB-VFI predicted im2 resized back to native",
    "oom": "Out of memory",
}

PRESENTED_PIPELINES = {
    "standard": ["vfi_middle"],
    "native": ["vfi_middle_back_to_native"],
}


def prettify_variant_name(variant: str) -> str:
    if variant == "BICUBIC":
        return "Bicubic baseline"
    return variant.replace("SRGAN_ResNet_", "").replace("_", " ")


def summarize_results(
    df: pd.DataFrame,
    pipelines=None,
    group_cols=("variant", "pipeline"),
):
    plot_df = df.copy()

    if pipelines is not None:
        plot_df = plot_df.loc[plot_df["pipeline"].isin(pipelines)].copy()

    plot_df = plot_df.loc[plot_df["pipeline"] != "oom"].copy()

    summary = (
        plot_df.groupby(list(group_cols), dropna=False)[["psnr", "ssim"]]
               .agg(["mean", "std", "count"])
               .reset_index()
    )
    summary.columns = list(group_cols) + [
        "psnr_mean", "psnr_std", "psnr_count",
        "ssim_mean", "ssim_std", "ssim_count",
    ]

    if "variant" in summary.columns:
        summary["variant_label"] = summary["variant"].map(prettify_variant_name)
    if "pipeline" in summary.columns:
        summary["pipeline_label"] = summary["pipeline"].map(PIPELINE_DISPLAY_NAMES).fillna(summary["pipeline"])

    sort_cols = [c for c in ["pipeline_label", "scale", "psnr_mean"] if c in summary.columns]
    ascending = [True] * max(len(sort_cols) - 1, 0) + [False] if sort_cols else True
    if sort_cols:
        summary = summary.sort_values(sort_cols, ascending=ascending).reset_index(drop=True)

    return summary


standard_summary = summarize_results(
    standard_df,
    pipelines=PRESENTED_PIPELINES["standard"],
    group_cols=("variant", "pipeline"),
)
standard_summary.to_csv(STANDARD_RESULTS_ROOT / "summary_metrics.csv", index=False)

display(
    standard_summary[[
        "variant_label", "pipeline_label",
        "psnr_mean", "psnr_std", "psnr_count",
        "ssim_mean", "ssim_std", "ssim_count",
    ]].rename(columns={"variant_label": "model"})
)


In [ ]:
def plot_metric_bars(summary_df: pd.DataFrame, title_prefix: str, facet_col: str | None = None):
    metric_specs = [
        ("psnr_mean", "PSNR", "{:.3f}"),
        ("ssim_mean", "SSIM", "{:.4f}"),
    ]

    if facet_col is None:
        facet_values = [None]
    else:
        facet_values = list(summary_df[facet_col].drop_duplicates())

    for metric_col, metric_label, value_fmt in metric_specs:
        for facet_value in facet_values:
            if facet_col is None:
                plot_df = summary_df.copy()
                facet_suffix = ""
            else:
                plot_df = summary_df.loc[summary_df[facet_col] == facet_value].copy()
                facet_suffix = f" | {facet_col}={facet_value}"

            if plot_df.empty:
                continue

            plot_df = plot_df.sort_values(metric_col, ascending=True)
            y_labels = plot_df["variant_label"] if "variant_label" in plot_df.columns else plot_df["variant"]

            fig_h = max(4.5, 0.65 * len(plot_df) + 1.5)
            fig, ax = plt.subplots(figsize=(12, fig_h))

            bars = ax.barh(y_labels, plot_df[metric_col].values)
            ax.set_xlabel(metric_label)
            ax.set_title(f"{title_prefix}: mean {metric_label}{facet_suffix}", fontsize=15, pad=12)
            ax.grid(axis="x", alpha=0.25)
            ax.set_axisbelow(True)

            xmin = float(plot_df[metric_col].min())
            xmax = float(plot_df[metric_col].max())
            span = max(xmax - xmin, 1e-6)
            ax.set_xlim(xmin - 0.08 * span, xmax + 0.18 * span)

            for bar, val in zip(bars, plot_df[metric_col].values):
                ax.text(
                    val + 0.02 * span,
                    bar.get_y() + bar.get_height() / 2,
                    value_fmt.format(val),
                    va="center",
                    fontsize=10,
                )

            plt.tight_layout()
            plt.show()


plot_metric_bars(standard_summary, "Standard x4 VFI-only benchmark")


In [ ]:
from matplotlib.colors import LinearSegmentedColormap

RED_YELLOW_ERROR_CMAP = LinearSegmentedColormap.from_list(
    "red_yellow_error",
    ["#5a0000", "#b30000", "#ff3b00", "#ff8c00", "#ffd400", "#fff176"],
)


def load_saved_variant_outputs(base_root: Path, variant: str, triplet_name: str):
    base = Path(base_root) / variant / triplet_output_name(triplet_name)

    # Backward-compatible fallback for results saved with the raw triplet name.
    if not base.exists():
        legacy_base = Path(base_root) / variant / normalize_triplet_index(triplet_name)
        if legacy_base.exists():
            base = legacy_base

    return {
        "im1_sr": Image.open(base / "im1_sr.png").convert("RGB"),
        "im2_sr": Image.open(base / "im2_sr.png").convert("RGB"),
        "im3_sr": Image.open(base / "im3_sr.png").convert("RGB"),
        "vfi_mid": Image.open(base / "vfi_mid.png").convert("RGB"),
    }


def pil_abs_diff_heatmap_normalized(pred: Image.Image, gt: Image.Image):
    pred_np = pil_to_np_uint8(pred).astype(np.float32)
    gt_np = pil_to_np_uint8(gt).astype(np.float32)
    if pred_np.shape != gt_np.shape:
        raise ValueError(f"Shape mismatch: pred={pred_np.shape}, gt={gt_np.shape}")
    diff = np.abs(pred_np - gt_np).mean(axis=2) / 255.0
    return diff


def _normalize_variants_argument(variants):
    if variants is None:
        return list(SRGAN_MODELS.keys())
    if isinstance(variants, str):
        return [variants]
    return list(variants)


def show_associated_lr_images(
    triplet_name: str,
    triplet_root: Path = TRIPLET_ROOT,
    downsample_factor: int = STANDARD_DOWNSAMPLE_FACTOR,
    show_native_size: bool = False,
):
    triplet_name = normalize_triplet_index(triplet_name)
    hr_triplet_dir = triplet_dir_from_index(triplet_name, triplet_root)
    hr1, hr2, hr3 = load_triplet_images(hr_triplet_dir)

    lr1 = pil_downsample(hr1, downsample_factor)
    lr2 = pil_downsample(hr2, downsample_factor)
    lr3 = pil_downsample(hr3, downsample_factor)

    if show_native_size:
        display_imgs = [
            pil_resize(lr1, (hr1.height, hr1.width)),
            pil_resize(lr2, (hr2.height, hr2.width)),
            pil_resize(lr3, (hr3.height, hr3.width)),
        ]
        suffix = "shown at HR display size"
    else:
        display_imgs = [lr1, lr2, lr3]
        suffix = "native LR size"

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    fig.suptitle(
        f"Associated LR inputs | triplet {triplet_name} | x{downsample_factor} downsample ({suffix})",
        fontsize=15
    )

    titles = [
        "LR input im1",
        "LR input im2",
        "LR input im3",
    ]

    for ax, img, title in zip(axes, display_imgs, titles):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis("off")

    plt.tight_layout()
    plt.show()


def show_standard_comparison(
    triplet_name: str,
    variants=None,
    triplet_root: Path = TRIPLET_ROOT,
    result_root: Path = STANDARD_RESULTS_ROOT,
):
    triplet_name = normalize_triplet_index(triplet_name)
    hr_triplet_dir = triplet_dir_from_index(triplet_name, triplet_root)
    hr1, hr2, hr3 = load_triplet_images(hr_triplet_dir)
    variants = _normalize_variants_argument(variants)

    for variant in variants:
        pred = load_saved_variant_outputs(result_root, variant, triplet_name)

        if variant.upper() == "BICUBIC":
            row2_titles = [
                "Bicubic upsampled im1",
                "Bicubic + TLB-VFI predicted im2",
                "Bicubic upsampled im3",
            ]
            middle_img = pred["vfi_mid"]
        else:
            row2_titles = [
                "SR im1",
                "SR + TLB-VFI predicted im2",
                "SR im3",
            ]
            middle_img = pred["vfi_mid"]

        metrics = {
            row2_titles[0]: compute_rgb_metrics(pred["im1_sr"], hr1),
            row2_titles[1]: compute_rgb_metrics(middle_img, hr2),
            row2_titles[2]: compute_rgb_metrics(pred["im3_sr"], hr3),
        }

        fig, axes = plt.subplots(2, 3, figsize=(16, 8))
        fig.suptitle(f"{prettify_variant_name(variant)} | Standard x4 comparison | triplet {triplet_name}", fontsize=16)

        gt_titles = [
            "HR Ground Truth im1",
            "HR Ground Truth im2",
            "HR Ground Truth im3",
        ]
        gt_imgs = [hr1, hr2, hr3]
        pred_imgs = [pred["im1_sr"], middle_img, pred["im3_sr"]]

        for ax, img, title in zip(axes[0], gt_imgs, gt_titles):
            ax.imshow(img)
            ax.set_title(title)
            ax.axis("off")

        for ax, img, title in zip(axes[1], pred_imgs, row2_titles):
            ax.imshow(img)
            ax.set_title(title)
            ax.axis("off")

        plt.tight_layout()
        plt.show()

        print(f"{prettify_variant_name(variant)} | triplet {triplet_name}")
        for label, (psnr, ssim) in metrics.items():
            print(f"  {label:<28} PSNR={psnr:.3f} | SSIM={ssim:.4f}")

        heatmaps = [
            pil_abs_diff_heatmap_normalized(pred["im1_sr"], hr1),
            pil_abs_diff_heatmap_normalized(middle_img, hr2),
            pil_abs_diff_heatmap_normalized(pred["im3_sr"], hr3),
        ]
        heat_titles = [
            "|HR Ground Truth im1 - SR im1|",
            "|HR Ground Truth im2 - (SR + TLB-VFI predicted im2)|",
            "|HR Ground Truth im3 - SR im3|",
        ] if variant.upper() != "BICUBIC" else [
            "|HR Ground Truth im1 - Bicubic im1|",
            "|HR Ground Truth im2 - (Bicubic + TLB-VFI predicted im2)|",
            "|HR Ground Truth im3 - Bicubic im3|",
        ]

        fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), constrained_layout = True)
        fig.suptitle(
            f"{prettify_variant_name(variant)} | Absolute pixel error heatmaps | triplet {triplet_name}",
            fontsize=16
        )

        im = None
        for ax, heat, title in zip(axes, heatmaps, heat_titles):
            im = ax.imshow(heat, cmap=RED_YELLOW_ERROR_CMAP, vmin=0.0, vmax=1.0)
            ax.set_title(title)
            ax.axis("off")

        cbar = fig.colorbar(im, ax=axes.ravel().tolist(), fraction=0.03, pad=0.02)
        cbar.set_label("Mean absolute pixel error (0 to 1)")
      #  plt.tight_layout()
        plt.show()


In [ ]:
def show_vfi_loss_grid(
    triplet_name: str,
    variants=None,
    triplet_root: Path = TRIPLET_ROOT,
    result_root: Path = STANDARD_RESULTS_ROOT,
    save_path=None,
    figsize=(8.5, 11),
    dpi=300,
    show=True,
):
    """
    Display a 4 x 2 thesis-sized figure comparing the HR middle-frame ground truth
    against saved TLB-VFI interpolated middle frames for available SRGAN variants.

    Layout:
        HR Ground Truth        MSE Loss
        L1 Loss                G-Loss
        SSIM Loss              MS-SSIM Loss
        MS-SSIM + L1 Loss      Huber Loss
    """

    def label_from_variant(variant):
        """
        Convert saved variant folder/key names into the thesis figure labels.
        Handles names like:
            SRGAN_ResNet_MSE
            SRGAN_ResNet_GLOSS
            SRGAN_ResNet_SSIM-LOSS
            SRGAN_ResNet_MS-SSIM
            SRGAN_ResNet_MS-SSIM + L1
        """
        v = str(variant).upper()

        # Check the most specific cases first.
        if "MS-SSIM + L1" in v or "MS_SSIM_L1" in v or "MSSSIML1" in v:
            return "MS-SSIM + L1 Loss"

        if "MS-SSIM" in v or "MS_SSIM" in v or "MSSSIM" in v:
            return "MS-SSIM Loss"

        if "SSIM" in v:
            return "SSIM Loss"

        if "GLOSS" in v or "G_LOSS" in v or "G-LOSS" in v:
            return "G-Loss"

        if "HUBER" in v:
            return "Huber Loss"

        if "MSE" in v:
            return "MSE Loss"

        # Check L1 late so MS-SSIM + L1 does not get mislabeled.
        if "L1" in v:
            return "L1 Loss"

        return None

    plot_order = [
        "HR Ground Truth",
        "MSE Loss",
        "L1 Loss",
        "G-Loss",
        "SSIM Loss",
        "MS-SSIM Loss",
        "MS-SSIM + L1 Loss",
        "Huber Loss",
    ]

    triplet_name = normalize_triplet_index(triplet_name)

    # Load HR triplet and keep only the middle HR frame.
    hr_triplet_dir = triplet_dir_from_index(triplet_name, triplet_root)
    _, hr2, _ = load_triplet_images(hr_triplet_dir)

    # Stay within available input variants.
    if variants is None:
        variants = list(SRGAN_MODELS.keys())
    elif isinstance(variants, str):
        variants = [variants]
    else:
        variants = list(variants)

    label_to_variant = {}

    for variant in variants:
        label = label_from_variant(variant)
        if label is not None:
            label_to_variant[label] = variant

    images = {"HR Ground Truth": hr2}
    used_variants = {}

    for label in plot_order:
        if label == "HR Ground Truth":
            continue

        if label not in label_to_variant:
            print(f"Skipping {label}: no available variant matched.")
            continue

        variant = label_to_variant[label]
        pred = load_saved_variant_outputs(result_root, variant, triplet_name)

        images[label] = pred["vfi_mid"]
        used_variants[label] = variant

    # Keep only labels that are actually available, plus HR ground truth.
    available_plot_labels = [
        label for label in plot_order
        if label in images
    ]

    n_panels = len(available_plot_labels)

    if n_panels == 8:
        nrows, ncols = 4, 2
    else:
        ncols = 2
        nrows = int(np.ceil(n_panels / ncols))

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=figsize,
        dpi=dpi,
        constrained_layout=True,
    )

    axes = np.array(axes).ravel()

    for ax, label in zip(axes, available_plot_labels):
        ax.imshow(images[label])
        ax.set_title(label, fontsize=12)
        ax.axis("off")

    # Hide unused axes if any variants are missing.
    for ax in axes[len(available_plot_labels):]:
        ax.axis("off")

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=dpi, bbox_inches="tight")
        print(f"Saved figure to: {save_path}")

    if show:
        plt.show()

    print("Saved VFI variants used:")
    for label in plot_order:
        if label in used_variants:
            print(f"  {label}: {used_variants[label]}")

    return fig, axes

In [ ]:
example_triplet = triplet_index_from_dir(TRIPLET_DIRS[138])

In [ ]:
fig, axes = show_vfi_loss_grid(
    triplet_name=example_triplet,
    #save_path="figures/vfi_loss_grid_triplet_00001.pdf",
)

## View the associated LR inputs for one triplet

This cell shows the three **x4 low-resolution input frames** used by the standard benchmark.
Use `show_native_size=True` if you want the LR frames enlarged to HR display size for easier viewing.


In [ ]:
example_triplet = triplet_index_from_dir(TRIPLET_DIRS[0])
show_associated_lr_images(example_triplet, show_native_size=True)


## Standard x4 comparison strips for each SR model + TLB-VFI pipeline

For each selected SR model, this cell shows only the **VFI-based pipeline**:

1. **Comparison strip**
   - Row 1: `HR Ground Truth im1`, `HR Ground Truth im2`, `HR Ground Truth im3`
   - Row 2: `SR im1`, `SR + TLB-VFI predicted im2`, `SR im3`

2. **Absolute-error heatmaps**
   - `|HR Ground Truth im1 - SR im1|`
   - `|HR Ground Truth im2 - (SR + TLB-VFI predicted im2)|`
   - `|HR Ground Truth im3 - SR im3|`

PSNR and SSIM for the row-2 images are printed directly below each comparison strip.


In [ ]:
example_triplet = triplet_index_from_dir(TRIPLET_DIRS[0])
show_standard_comparison(example_triplet)  # iterates through all loaded SRGAN models


In [ ]:
for i_ in range(1,100):
  example_triplet = triplet_index_from_dir(TRIPLET_DIRS[i_])
  show_associated_lr_images(example_triplet, show_native_size=True)
  show_standard_comparison(example_triplet)

# Native Upscale

In [ ]:
def run_native_upscale_benchmark(
    triplet_dirs,
    srgan_models,
    tlb_model,
    results_root: Path,
    target_scales=(4, 16),
    max_triplets=10,
    save_images: bool = True,
    force_rerun: bool = False,
):
    results_root = Path(results_root)
    csv_path = results_root / "per_triplet_metrics.csv"

    if csv_path.exists() and not force_rerun:
        print(f"Loading existing results from {csv_path}")
        return pd.read_csv(csv_path)

    subset = triplet_dirs[:max_triplets]
    rows = []

    for total_scale in target_scales:
        recursive_passes = recursive_passes_for_total_scale(total_scale, base_scale=4)

        for triplet_dir in tqdm(subset, desc=f"Native-upscale benchmark x{total_scale}"):
            triplet_name = triplet_index_from_dir(triplet_dir)
            output_name = triplet_output_name(triplet_name)

            hr1, hr2, hr3 = load_triplet_images(triplet_dir)
            gt_size_hw = (hr2.height, hr2.width)

            # Bicubic baseline
            bic1 = bicubic_upscale_pil(hr1, total_scale)
            bic2 = bicubic_upscale_pil(hr2, total_scale)
            bic3 = bicubic_upscale_pil(hr3, total_scale)

            try:
                bic_vfi_big = tlbvfi_interpolate_pil(bic1, bic3, tlb_model, device=device)
                bic2_back = pil_resize(bic2, gt_size_hw)
                bic_vfi_back = pil_resize(bic_vfi_big, gt_size_hw)

                if save_images:
                    save_dir = results_root / f"x{total_scale}" / "BICUBIC" / output_name
                    save_pil(bic1, save_dir / "im1_sr.png")
                    save_pil(bic2, save_dir / "im2_sr.png")
                    save_pil(bic3, save_dir / "im3_sr.png")
                    save_pil(bic_vfi_big, save_dir / "vfi_mid_big.png")
                    save_pil(bic2_back, save_dir / "im2_back_to_native.png")
                    save_pil(bic_vfi_back, save_dir / "vfi_mid_back_to_native.png")

                psnr, ssim = compute_rgb_metrics(bic2_back, hr2)
                rows.append({
                    "scale": total_scale,
                    "triplet": triplet_name,
                    "variant": "BICUBIC",
                    "pipeline": "direct_middle_back_to_native",
                    "compare_mode": "resized_to_native",
                    "psnr": psnr,
                    "ssim": ssim,
                })

                psnr, ssim = compute_rgb_metrics(bic_vfi_back, hr2)
                rows.append({
                    "scale": total_scale,
                    "triplet": triplet_name,
                    "variant": "BICUBIC",
                    "pipeline": "vfi_middle_back_to_native",
                    "compare_mode": "resized_to_native",
                    "psnr": psnr,
                    "ssim": ssim,
                })
            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    clear_cuda()
                    rows.append({
                        "scale": total_scale,
                        "triplet": triplet_name,
                        "variant": "BICUBIC",
                        "pipeline": "oom",
                        "compare_mode": "resized_to_native",
                        "psnr": np.nan,
                        "ssim": np.nan,
                    })
                    continue
                raise

            for model_name, generator in srgan_models.items():
                try:
                    sr1_big = srgan_super_resolve_pil(hr1, generator, recursive_passes=recursive_passes, device=device)
                    sr2_big = srgan_super_resolve_pil(hr2, generator, recursive_passes=recursive_passes, device=device)
                    sr3_big = srgan_super_resolve_pil(hr3, generator, recursive_passes=recursive_passes, device=device)
                    vfi_big = tlbvfi_interpolate_pil(sr1_big, sr3_big, tlb_model, device=device)

                    sr2_back = pil_resize(sr2_big, gt_size_hw)
                    vfi_back = pil_resize(vfi_big, gt_size_hw)

                    if save_images:
                        save_dir = results_root / f"x{total_scale}" / model_name / output_name
                        save_pil(sr1_big, save_dir / "im1_sr.png")
                        save_pil(sr2_big, save_dir / "im2_sr.png")
                        save_pil(sr3_big, save_dir / "im3_sr.png")
                        save_pil(vfi_big, save_dir / "vfi_mid_big.png")
                        save_pil(sr2_back, save_dir / "im2_back_to_native.png")
                        save_pil(vfi_back, save_dir / "vfi_mid_back_to_native.png")

                    psnr, ssim = compute_rgb_metrics(sr2_back, hr2)
                    rows.append({
                        "scale": total_scale,
                        "triplet": triplet_name,
                        "variant": model_name,
                        "pipeline": "direct_middle_back_to_native",
                        "compare_mode": "resized_to_native",
                        "psnr": psnr,
                        "ssim": ssim,
                    })

                    psnr, ssim = compute_rgb_metrics(vfi_back, hr2)
                    rows.append({
                        "scale": total_scale,
                        "triplet": triplet_name,
                        "variant": model_name,
                        "pipeline": "vfi_middle_back_to_native",
                        "compare_mode": "resized_to_native",
                        "psnr": psnr,
                        "ssim": ssim,
                    })

                except RuntimeError as e:
                    if "out of memory" in str(e).lower():
                        clear_cuda()
                        rows.append({
                            "scale": total_scale,
                            "triplet": triplet_name,
                            "variant": model_name,
                            "pipeline": "oom",
                            "compare_mode": "resized_to_native",
                            "psnr": np.nan,
                            "ssim": np.nan,
                        })
                        print(f"OOM at native x{total_scale} / {triplet_name} / {model_name}")
                        continue
                    raise

    df = pd.DataFrame(rows)
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(csv_path, index=False)
    print(f"Saved native-upscale metrics to {csv_path}")
    return df


In [ ]:
if RUN_NATIVE_UPSCALE_SECTION:
    native_df = run_native_upscale_benchmark(
        triplet_dirs=TRIPLET_DIRS,
        srgan_models=SRGAN_MODELS,
        tlb_model=TLB_MODEL,
        results_root=NATIVE_RESULTS_ROOT,
        target_scales=NATIVE_TARGET_SCALES,
        max_triplets=NATIVE_MAX_TRIPLETS,
        save_images=SAVE_IMAGES,
        force_rerun=FORCE_RERUN,
    )

    native_summary = summarize_results(
        native_df,
        pipelines=PRESENTED_PIPELINES["native"],
        group_cols=("scale", "variant", "pipeline"),
    )
    native_summary.to_csv(NATIVE_RESULTS_ROOT / "summary_metrics.csv", index=False)

    display(
        native_summary[[
            "scale", "variant_label", "pipeline_label",
            "psnr_mean", "psnr_std", "psnr_count",
            "ssim_mean", "ssim_std", "ssim_count",
        ]].rename(columns={"variant_label": "model"})
    )

    plot_metric_bars(native_summary, "Native upscale VFI-only benchmark", facet_col="scale")
else:
    print("Native-upscale section is disabled. Set RUN_NATIVE_UPSCALE_SECTION = True to run it.")


In [ ]:
def show_native_comparison(
    scale: int,
    triplet_name: str,
    variant: str,
    triplet_root: Path = TRIPLET_ROOT,
    result_root: Path = NATIVE_RESULTS_ROOT,
):
    triplet_name = normalize_triplet_index(triplet_name)
    hr1, hr2, hr3 = load_triplet_images(triplet_dir_from_index(triplet_name, triplet_root))

    base = Path(result_root) / f"x{scale}" / variant / triplet_output_name(triplet_name)
    if not base.exists():
        legacy_base = Path(result_root) / f"x{scale}" / variant / triplet_name
        if legacy_base.exists():
            base = legacy_base

    vfi_middle_back = Image.open(base / "vfi_mid_back_to_native.png").convert("RGB")

    heat_vfi_middle = pil_abs_diff_heatmap_normalized(vfi_middle_back, hr2)
    psnr, ssim = compute_rgb_metrics(vfi_middle_back, hr2)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
    fig.suptitle(
        f"Native x{scale} | {prettify_variant_name(variant)} | VFI-only comparison | triplet {triplet_name}",
        fontsize=16
    )

    axes[0].imshow(hr2)
    axes[0].set_title("HR Ground Truth im2")

    axes[1].imshow(vfi_middle_back)
    axes[1].set_title("Upscaled SR + TLB-VFI predicted im2 resized back to native")

    heat_im = axes[2].imshow(heat_vfi_middle, cmap=RED_YELLOW_ERROR_CMAP, vmin=0.0, vmax=1.0)
    axes[2].set_title("|HR Ground Truth im2 - (Upscaled SR + TLB-VFI predicted im2)|")

    for ax in axes:
        ax.axis("off")

    cbar = fig.colorbar(heat_im, ax=axes[2], fraction=0.046, pad=0.04)
    cbar.set_label("Mean absolute pixel error (0 to 1)")

    plt.tight_layout()
    plt.show()

    print(f"{prettify_variant_name(variant)} | native x{scale} | triplet {triplet_name}")
    print(f"  Upscaled SR + TLB-VFI predicted im2 resized back to native  PSNR={psnr:.3f} | SSIM={ssim:.4f}")

# Example after running the native section:
# show_native_comparison(scale=4, triplet_name=triplet_index_from_dir(TRIPLET_DIRS[0]), variant=list(SRGAN_MODELS.keys())[0])


In [ ]:
print("Standard results saved under:", STANDARD_RESULTS_ROOT)
print("Native results saved under  :", NATIVE_RESULTS_ROOT)
print("Standard CSV               :", STANDARD_RESULTS_ROOT / "per_triplet_metrics.csv")
print("Standard summary CSV       :", STANDARD_RESULTS_ROOT / "summary_metrics.csv")
if RUN_NATIVE_UPSCALE_SECTION:
    print("Native CSV                 :", NATIVE_RESULTS_ROOT / "per_triplet_metrics.csv")
    print("Native summary CSV         :", NATIVE_RESULTS_ROOT / "summary_metrics.csv")